In [ ]:
base_path = "/Volumes/workspace/default_capstone/capstone_files/"

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "sales.csv")
)

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "products.csv")
)

display(sales_df)
display(products_df)

In [ ]:
print("SALES DATA")
sales_df.printSchema()

print("PRODUCTS DATA")
products_df.printSchema()

In [ ]:
display(sales_df.limit(5))
display(products_df.limit(5))

## 2. Add the additional product and sales records

In [ ]:
new_products = spark.createDataFrame(
    [
        (204, "Mouse", "Accessories"),
        (205, "Monitor", "Electronics"),
        (206, "Printer", "Electronics")
    ],
    ["product_id", "product_name", "category"]
)

products_df = products_df.unionByName(new_products)

display(products_df)

In [ ]:
new_sales = spark.createDataFrame(
    [
        (6, 104, 204, 15, 500, 250, 30),
        (7, 104, 205, 6, 8000, 5500, 200),
        (8, 105, 206, 4, 6000, 4000, 100),
        (9, 105, 201, 3, 500, 300, 10),
        (10, 106, 202, 8, 1000, 650, 40),
        (11, 106, 204, 10, 500, 250, 20)
    ],
    ["sale_id", "store_id", "product_id", "quantity",
     "price", "cost", "discount"]
)

sales_df = sales_df.unionByName(new_sales)

display(sales_df)

In [ ]:
print("Total sales records:", sales_df.count())
print("Total products:", products_df.count())

## 3. Expand the sample sales dataset

In [ ]:
from pyspark.sql.functions import *

large_sales = (
    spark.range(17, 50001)
    .withColumnRenamed("id", "sale_id")
    .withColumn("store_id", (col("sale_id") % 10 + 101).cast("int"))
    .withColumn("product_id", (col("sale_id") % 6 + 201).cast("int"))
    .withColumn("quantity", (col("sale_id") % 15 + 1).cast("int"))
    .withColumn(
        "price",
        when(col("product_id") == 201, 500)
        .when(col("product_id") == 202, 1000)
        .when(col("product_id") == 203, 1500)
        .when(col("product_id") == 204, 500)
        .when(col("product_id") == 205, 8000)
        .otherwise(6000)
    )
    .withColumn(
        "cost",
        when(col("product_id") == 201, 300)
        .when(col("product_id") == 202, 650)
        .when(col("product_id") == 203, 1000)
        .when(col("product_id") == 204, 250)
        .when(col("product_id") == 205, 5500)
        .otherwise(4000)
    )
    .withColumn("discount", (col("quantity") * 5).cast("int"))
    .select(
        "sale_id", "store_id", "product_id",
        "quantity", "price", "cost", "discount"
    )
)

sales_df = (
    sales_df.unionByName(large_sales)
    .dropDuplicates(["sale_id"])
)

display(sales_df)

In [ ]:
print("Total Sales Records:", sales_df.count())
print("Total Products:", products_df.count())

In [ ]:
from pyspark.sql.functions import *

new_products = (
    spark.range(207, 701)
    .withColumnRenamed("id", "product_id")
    .withColumn(
        "product_name",
        concat(lit("Product_"), col("product_id"))
    )
    .withColumn(
        "category",
        when(col("product_id") % 4 == 0, "Electronics")
        .when(col("product_id") % 4 == 1, "Accessories")
        .when(col("product_id") % 4 == 2, "Furniture")
        .otherwise("Stationery")
    )
    .select("product_id", "product_name", "category")
)

products_df = products_df.unionByName(new_products)

display(products_df)

## 4. Data quality checks

In [ ]:
from pyspark.sql.functions import *

quality_report = sales_df.agg(
    count("*").alias("total_records"),
    countDistinct("sale_id").alias("unique_sales"),
    sum(when(col("sale_id").isNull(), 1).otherwise(0)).alias("missing_sale_ids"),
    sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("missing_product_ids"),
    sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("missing_quantities"),
    sum(when(col("price") <= 0, 1).otherwise(0)).alias("invalid_prices"),
    sum(when(col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantities")
)

display(quality_report)

In [ ]:
duplicate_sales = (
    sales_df
    .groupBy("sale_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_sales)

## 5. Transform and join product and sales data

In [ ]:
from pyspark.sql.functions import *

product_sales = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount"),
        count("sale_id").alias("total_orders"),
        avg("price").alias("average_selling_price"),
        avg("cost").alias("average_cost")
    )
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "profit_margin",
        when(col("net_revenue") != 0,
             round(col("profit") / col("net_revenue") * 100, 2)
        ).otherwise(0)
    )
)

# Left join keeps every product, including products with no sales.
final_product_metrics = (
    products_df
    .join(product_sales, on="product_id", how="left")
    .fillna({
        "total_quantity_sold": 0,
        "gross_revenue": 0,
        "total_cost": 0,
        "total_discount": 0,
        "total_orders": 0,
        "average_selling_price": 0,
        "average_cost": 0,
        "net_revenue": 0,
        "profit": 0,
        "profit_margin": 0
    })
)

display(final_product_metrics)


## 6. Category-wise profit margin

In [ ]:
category_metrics = (
    final_product_metrics
    .groupBy("category")
    .agg(
        sum("total_orders").alias("total_orders"),
        sum("total_quantity_sold").alias("total_quantity_sold"),
        sum("gross_revenue").alias("gross_revenue"),
        sum("total_discount").alias("total_discount"),
        sum("net_revenue").alias("net_revenue"),
        sum("total_cost").alias("total_cost"),
        sum("profit").alias("total_profit")
    )
    .withColumn(
        "profit_margin",
        when(col("net_revenue") != 0,
             round(col("total_profit") / col("net_revenue") * 100, 2)
        ).otherwise(0)
    )
)
display(category_metrics.orderBy(desc("total_profit")))


## 7. Store performance metrics

In [ ]:
store_metrics = (
    sales_df
    .groupBy("store_id")
    .agg(
        count("sale_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        sum("discount").alias("total_discount")
    )
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "profit_margin",
        when(col("net_revenue") != 0,
             round(col("profit") / col("net_revenue") * 100, 2)
        ).otherwise(0)
    )
)
display(store_metrics.orderBy(desc("net_revenue")))


## 8. Discount analysis

In [ ]:
discount_analysis = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("discount").alias("total_discount"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum(col("quantity") * col("cost")).alias("total_cost"),
        count("sale_id").alias("total_orders")
    )
    .withColumn("net_revenue", col("gross_revenue") - col("total_discount"))
    .withColumn("profit", col("net_revenue") - col("total_cost"))
    .withColumn(
        "discount_percentage",
        when(col("gross_revenue") != 0,
             round(col("total_discount") / col("gross_revenue") * 100, 2)
        ).otherwise(0)
    )
)
display(discount_analysis)


## 9. Daily sales metrics

In [ ]:
from pyspark.sql.functions import *

sales_with_date = (
    sales_df
    .withColumn(
        "sale_date",
        date_add(
            to_date(lit("2026-10-01")),
            (col("sale_id") % 30).cast("int")
        )
    )
)

daily_sales = (
    sales_with_date
    .groupBy("sale_date")
    .agg(
        count("sale_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum(col("quantity") * col("price")).alias("gross_revenue"),
        sum("discount").alias("total_discount")
    )
    .withColumn(
        "net_revenue",
        col("gross_revenue") - col("total_discount")
    )
    .orderBy("sale_date")
)

display(daily_sales)

## 10. Product ranking and low-selling products

In [ ]:
from pyspark.sql.window import Window

product_window = Window.orderBy(desc("total_quantity_sold"))

product_ranking = (
    final_product_metrics
    .withColumn("sales_rank", dense_rank().over(product_window))
)

display(product_ranking)

In [ ]:
low_selling_products = (
    final_product_metrics
    .filter(col("total_quantity_sold") < 100)
    .orderBy("total_quantity_sold")
)

display(low_selling_products)

## 11. Running revenue and moving average

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

revenue_window = (
    Window
    .orderBy("sale_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_revenue = (
    daily_sales
    .withColumn(
        "cumulative_revenue",
        sum("net_revenue").over(revenue_window)
    )
)

display(running_revenue)

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

moving_window = (
    Window
    .orderBy("sale_date")
    .rowsBetween(-6, 0)
)

moving_average = (
    daily_sales
    .withColumn(
        "moving_average_revenue",
        round(avg("net_revenue").over(moving_window), 2)
    )
)

display(moving_average)

## 12. Save dashboard-ready Delta tables

In [ ]:
final_product_metrics.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_product_metrics")
category_metrics.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_category_metrics")
store_metrics.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_store_metrics")
discount_analysis.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_discount_analysis")
daily_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_daily_sales")
product_ranking.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_product_ranking")
low_selling_products.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_low_selling_products")
running_revenue.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_running_revenue")
moving_average.write.format("delta").mode("overwrite").saveAsTable("workspace.default.retail_moving_average")

print("All dashboard Delta tables saved successfully.")


## 13. Databricks SQL – Top 3 best-selling products

In [ ]:
%sql
SELECT
    product_id,
    product_name,
    category,
    total_quantity_sold,
    net_revenue,
    profit,
    profit_margin
FROM workspace.default.retail_product_metrics
ORDER BY total_quantity_sold DESC, product_id
LIMIT 3;


## 14. Verify saved outputs

In [ ]:
%sql
SHOW TABLES IN workspace.default;


In [ ]:
%sql
SELECT category, total_quantity_sold, net_revenue, total_profit, profit_margin
FROM workspace.default.retail_category_metrics
ORDER BY total_profit DESC;


In [ ]:
%sql
SELECT *
FROM workspace.default.retail_product_metrics
ORDER BY total_quantity_sold DESC
LIMIT 10;


## 15. Final validation and output summary

Run this section after all ETL and save-table cells. The displayed results are generated by Databricks when the notebook is executed; they are not sample or fabricated values.

In [ ]:
print("WEEK 4 ETL PIPELINE - FINAL CHECK")
print("=" * 45)
print("Sales records:", sales_df.count())
print("Product records:", products_df.count())
print("Product metric records:", final_product_metrics.count())
print("Category metric records:", category_metrics.count())
print("Store metric records:", store_metrics.count())
print("\nDashboard Delta tables:")
for table_name in [
    "retail_product_metrics",
    "retail_category_metrics",
    "retail_store_metrics",
    "retail_discount_analysis",
    "retail_daily_sales",
    "retail_product_ranking",
    "retail_low_selling_products",
    "retail_running_revenue",
    "retail_moving_average"
]:
    print("-", table_name)
print("\nETL validation cell completed. Check the SQL results above for the top 3 products.")